In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA probe directions through J-Lens

This descriptive experiment connects the already-trained FLenQA probes to the existing J-Lens Jacobians. For each held-out 2000- and 3000-token example, it measures how the gold-oriented probe direction propagates to the final hidden state and to the correct-vs-incorrect answer-logit margin. It does not edit hidden states or run interventions.


In [ ]:
%pip install -qq --disable-pip-version-check pandas matplotlib


In [ ]:
import json

import jlens
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.evaluation import evaluate_paper_binary
from jlens_reasoning.evaluation_utils import answer_token_variants

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
PROBE_PATH = ASSET_DIR / "probes.pt"
METADATA_PATH = ASSET_DIR / "metadata.json"
SPLIT_PATH = ASSET_DIR / "problem_split.json"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
RESULT_PATH = context.runs_dir / "flenqa-probe-jlens" / "propagation.parquet"
EXPECTED_CONTEXT_SIZES = (2000, 3000)
HEADLINE_LAYER = 18
MAX_SEQ_LEN = 4096


## Load frozen probes, model, and J-Lens

The probe weights and validation metrics come from the probe-assets notebook. The held-out model correctness comes from the existing FLenQA model-output artifact. The Jacobian is used only through J @ d, so no example-level full Jacobian is constructed or saved.

In [ ]:
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
checkpoint = torch.load(PROBE_PATH, map_location="cpu", weights_only=False)
assert metadata["format_version"] == checkpoint["format_version"] == 1
assert metadata["split"] == split_asset
test_ids = set(split_asset["problems"]["test"])

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
causal_lm.eval()
num_layers = int(causal_lm.config.num_hidden_layers)
hidden_dim = int(causal_lm.config.hidden_size)
assert metadata["num_layers"] == num_layers
assert metadata["hidden_dim"] == hidden_dim
assert set(checkpoint["layers"]) == set(range(num_layers))

lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
assert set(lens.source_layers) == set(range(num_layers))
jacobians = {
    layer: lens.jacobians[layer].detach().float().cpu()
    for layer in sorted(lens.source_layers)
}
assert all(matrix.shape == (hidden_dim, hidden_dim) for matrix in jacobians.values())
unembedding = causal_lm.get_output_embeddings().weight.detach().float().cpu()
true_ids = tuple(token_id for token_id, _ in answer_token_variants(tokenizer, ("True",)))
false_ids = tuple(token_id for token_id, _ in answer_token_variants(tokenizer, ("False",)))
assert true_ids and false_ids
answer_direction = unembedding[list(true_ids)].mean(dim=0) - unembedding[list(false_ids)].mean(dim=0)
print({"model": MODEL_NAME, "layers": num_layers, "answer_true_ids": true_ids, "answer_false_ids": false_ids})


In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
test_rows = [row for row in rows if row.problem_id in test_ids]
test_prompts = prepare_prompts(test_rows)
prompt_context_sizes = {}
for prompt in test_prompts:
    sizes = {item.ctx_size for item in prompt.provenance}
    assert len(sizes) == 1
    prompt_context_sizes[prompt.prompt_id] = sizes.pop()
test_prompts = [prompt for prompt in test_prompts if prompt_context_sizes[prompt.prompt_id] in EXPECTED_CONTEXT_SIZES]
assert {prompt_context_sizes[p.prompt_id] for p in test_prompts} == set(EXPECTED_CONTEXT_SIZES)

model_records = {record["prompt_id"]: record for record in pq.read_table(MODEL_OUTPUT_PATH).to_pylist()}
examples = []
for prompt in test_prompts:
    record = model_records[prompt.prompt_id]
    evaluation = evaluate_paper_binary(record["generated_text"], expected=prompt.label)
    examples.append({
        "prompt_id": prompt.prompt_id,
        "problem_id": prompt.problem_id,
        "ctx_size": prompt_context_sizes[prompt.prompt_id],
        "task": prompt.task,
        "label": int(prompt.label),
        "model_correct": bool(evaluation.correct),
        "prompt": prompt,
    })
assert examples and len({item["problem_id"] for item in examples}) == len(test_ids)
pd.DataFrame(examples).groupby("ctx_size").size()


## Compute propagation and answer effects

For each layer, gold_probe_score is the raw probe score oriented toward the gold label. The normalized direction d is oriented the same way. answer_effect uses the output direction from the exact single-token True/False variants already used by project evaluation, with the direction oriented toward the gold answer.

In [ ]:
result_rows = []
for item in tqdm(examples, desc="Computing J-Lens probe propagation"):
    prompt = item["prompt"]
    encoded = tokenizer(prompt.text, return_tensors="pt", truncation=False)
    input_ids = encoded["input_ids"].to(context.device)
    assert 0 < input_ids.shape[1] <= MAX_SEQ_LEN
    with torch.inference_mode():
        outputs = causal_lm(input_ids=input_ids, output_hidden_states=True, use_cache=False)
    hidden_states = outputs.hidden_states
    assert hidden_states is not None and len(hidden_states) == num_layers + 1
    for layer in range(num_layers):
        asset = checkpoint["layers"][layer]
        hidden = hidden_states[layer + 1][0, -1, :].detach().float().cpu()
        raw_score = torch.dot(hidden - asset["training_mean"].float(), asset["weight"].float()) + asset["bias"].float()
        gold_sign = 1.0 if item["label"] == 1 else -1.0
        gold_probe_score = float(gold_sign * raw_score)
        direction = gold_sign * asset["unit_weight"].float()
        propagated = jacobians[layer].mv(direction)
        gold_output_direction = gold_sign * answer_direction
        result_rows.append({
            "prompt_id": item["prompt_id"],
            "problem_id": item["problem_id"],
            "ctx_size": item["ctx_size"],
            "task": item["task"],
            "label": item["label"],
            "model_correct": item["model_correct"],
            "layer": layer,
            "probe_score": float(raw_score),
            "gold_probe_score": gold_probe_score,
            "propagation_norm": float(torch.linalg.vector_norm(propagated)),
            "answer_effect": float(torch.dot(gold_output_direction, propagated)),
        })
results = pd.DataFrame(result_rows)
assert len(results) == len(examples) * num_layers
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
pq.write_table(pa.Table.from_pandas(results, preserve_index=False), RESULT_PATH)
display(results.head())
print(f"Saved {len(results):,} rows to {RESULT_PATH}")


## Layer summaries

These summaries are descriptive and keep the model-correct and model-wrong groups separate. Layer 18 is shown as the validation-selected headline layer, but no layer is selected using these held-out results.

In [ ]:
summary = (
    results.groupby(["layer", "model_correct"], as_index=False)
    .agg(
        mean_propagation_norm=("propagation_norm", "mean"),
        mean_answer_effect=("answer_effect", "mean"),
    )
)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for model_correct, group in summary.groupby("model_correct", sort=True):
    label = "model-correct" if model_correct else "model-wrong"
    axes[0].plot(group["layer"], group["mean_propagation_norm"], marker="o", label=label)
    axes[1].plot(group["layer"], group["mean_answer_effect"], marker="o", label=label)
axes[0].set_title("Gold-probe propagation norm")
axes[0].set_ylabel("mean |J d|")
axes[1].set_title("Gold-probe answer effect")
axes[1].set_ylabel("mean answer_effect")
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
for axis in axes:
    axis.set_xlabel("Layer")
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()


In [ ]:
wrong = results[(~results["model_correct"]) & (results["ctx_size"].isin(EXPECTED_CONTEXT_SIZES))].copy()
correlation = wrong["gold_probe_score"].corr(wrong["answer_effect"])
print({"headline_layer": HEADLINE_LAYER, "model_wrong_rows": len(wrong), "gold_probe_score_answer_effect_correlation": float(correlation)})


## Interpretation

1. **Is answer information still linearly decodable at long context?** Use the raw and gold-oriented probe scores/AUROC from the held-out probe-evaluation notebook; fixed-threshold probe accuracy is not sufficient because of threshold drift.
2. **Does the probe direction propagate toward the answer?** Compare propagation_norm and answer_effect across layers and model-correctness groups. Positive answer_effect is locally answer-aligned; negative values are answer-opposing; near-zero values indicate little first-order effect.
3. **Do probe evidence and answer effect covary on model failures?** The reported correlation is descriptive only. It does not establish that the decodable information is causally used, and this notebook performs no hidden-state editing or intervention.